# Qwen typed full LLM multi-agent TEC experiment

This notebook runs the typed-contract full LLM multi-agent network on the same five TEC tasks used by the Qwen single-agent and historical full LLM multi-agent notebooks. The typed protocol is an experimental interface contract, not a scripted role workflow.

## 1. Clean Colab setup

Run this in Google Colab first. After it finishes, restart the runtime before importing transformers. This notebook expects an already processed TEC parquet and does not rebuild IONEX data.

In [ ]:
# Clean Colab setup for Qwen.
# Run this cell first, then restart runtime before importing transformers.

!pip uninstall -y transformers scikit-learn scipy
!pip install -U accelerate bitsandbytes torchvision pillow sentencepiece protobuf safetensors huggingface_hub pandas pyarrow pydantic python-dateutil
!pip install "transformers @ git+https://github.com/huggingface/transformers.git@main"

IMPORTANT: after setup, restart the Colab runtime before importing transformers.

## 2. Import check

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 3. Clone or update repository

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "main"], check=True)

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print("Working directory:", Path.cwd())

## 4. Project imports

In [ ]:
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from tec_agents.agents.llm_multi_agent_typed import LLMFullTypedMultiAgent
from tec_agents.agents.llm_multi_agent_typed_prompts import ARCHITECTURE_NAME
from tec_agents.agents.llm_multi_agent_typed_protocol import TYPED_PROTOCOL_VERSION
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.five_task_configs import (
    get_five_task_configs,
    build_five_task_eval_task,
    build_five_task_expected_sequence,
)
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import MetricResult, compare_agent_to_gold
from tec_agents.eval.task_set import task_to_dict
from tec_agents.llm.local_qwen import LocalQwenChatModel
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server

## 5. CONFIG

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
TORCH_DTYPE = "float16"
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
DO_SAMPLE = False

DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
PROCESSED_DATASET_PATH = PROJECT_DIR / "data" / "processed" / DATASET_FILENAME
START = "2024-03-01"
END = "2024-04-01"

ARCHITECTURE_MODE = "qwen_multi_agent_typed_full_llm"
PROMPT_REVISION = "grounded_inputs_deliverables_single_block_v3"
EXPERIMENT_ID = "qwen_multi_agent_typed_v3_batch_colab"
MAX_ORCHESTRATION_STEPS = 12
MAX_ROLE_STEPS = 8
MAX_TOOL_CALLS_PER_ROLE = 8
MAX_PARSE_RETRIES = 2
MAX_TOOL_RETRIES = 2
STATE_FEEDBACK_MODE = "typed_state"

RUN_ALL_TASKS = True
SELECTED_PRESET = "hightec_midlat_europe"

OUTPUT_ROOT = PROJECT_DIR / "outputs" / "metrics"
EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / "experiment_4"
OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR
BATCH_OUTPUT_PATH = EXPERIMENT_OUTPUT_DIR / "qwen_multi_agent_typed_v3_batch_colab.json"
PER_TASK_OUTPUT_TEMPLATE = "qwen_multi_agent_typed_v3_{preset_id}_colab.json"

TASK_CONFIGS = get_five_task_configs(dataset_ref=DATASET_REF)
if RUN_ALL_TASKS:
    SELECTED_TASK_CONFIGS = TASK_CONFIGS
else:
    SELECTED_TASK_CONFIGS = [c for c in TASK_CONFIGS if c["preset_id"] == SELECTED_PRESET]
assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET={SELECTED_PRESET!r}"
print("Architecture:", ARCHITECTURE_MODE)
print("Typed protocol:", TYPED_PROTOCOL_VERSION)
print("Prompt revision:", PROMPT_REVISION)
print("Selected tasks:", [c["preset_id"] for c in SELECTED_TASK_CONFIGS])

## Planned test questions

This preview is for the human notebook reader only. The expected tool sequence is used later for evaluation and is not passed into the typed LLM agents.

In [ ]:
preview_rows = []
for config in SELECTED_TASK_CONFIGS:
    preview_rows.append({
        "preset_id": config["preset_id"],
        "task_type": config["task_type"],
        "query": config["query"],
        "expected_tool_sequence": " -> ".join(build_five_task_expected_sequence(config)),
    })
pd.DataFrame(preview_rows)

## 6. Dataset setup

In [ ]:
DATASET_PATH = PROCESSED_DATASET_PATH
assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"
register_dataset(dataset_ref=DATASET_REF, path=DATASET_PATH, file_format="parquet")
get_dataset_summary(DATASET_REF)

## 7. Qwen loading

In [ ]:
model = LocalQwenChatModel(
    model_name=MODEL_NAME,
    device_map="auto",
    torch_dtype=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=LOAD_IN_8BIT,
    trust_remote_code=True,
    max_input_tokens=MAX_INPUT_TOKENS,
)

## 8. Helpers

In [ ]:
def to_jsonable(value):
    if hasattr(value, "to_dict"):
        return to_jsonable(value.to_dict())
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if hasattr(value, "item"):
        return value.item()
    return value


def metric_dict(metric_result):
    if metric_result is None:
        return {}
    return metric_result.metrics if isinstance(metric_result, MetricResult) else dict(metric_result)


def final_answer_preview(result, limit=360):
    text = getattr(result, "answer", "") or ""
    return text[:limit] + ("..." if len(text) > limit else "")


def build_verdict_checks(result, metric_result):
    metrics = metric_dict(metric_result)
    return {
        "agent_success": bool(result.success),
        "tool_sequence_match": metrics.get("tool_sequence_match"),
        "final_answer_present": bool(result.answer),
        "no_parse_errors": result.parse_error_count == 0,
        "no_stall": not result.stalled_loop_detected,
        "no_forbidden_tool_calls": result.forbidden_tool_call_count == 0,
        "no_premature_role_completion": result.premature_role_completion_count == 0,
        "no_empty_analysis_findings": result.empty_findings_done_count == 0,
    }


def overall_ok_from_checks(checks):
    required = [v for v in checks.values() if isinstance(v, bool)]
    return bool(required) and all(required)


def key_metric_summary(task_type, metrics):
    if task_type == "high_tec":
        return f"threshold_abs_error={metrics.get('threshold_abs_error')}; interval_count_error={metrics.get('interval_count_error')}"
    if task_type == "stable_intervals":
        return f"stable_interval_count_error={metrics.get('stable_interval_count_error')}"
    if task_type == "compare_regions":
        return f"mean_abs_error_avg={metrics.get('mean_abs_error_avg')}; pairwise_delta_count_match={metrics.get('pairwise_delta_count_match')}"
    if task_type == "report":
        return f"required_artifacts_present={metrics.get('required_artifacts_present')}; grounded={metrics.get('report_grounded_in_artifacts')}"
    return ""


## 9. Batch loop

In [ ]:
all_results = []
gold_runner = GoldRunner()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for task_config in SELECTED_TASK_CONFIGS:
    preset_id = task_config["preset_id"]
    query = task_config["query"]
    expected_sequence = build_five_task_expected_sequence(task_config)
    eval_task = build_five_task_eval_task(task_config)

    print()
    print("===", preset_id, "===")
    server = build_local_mcp_server(run_id=f"qwen_multi_agent_typed_v3_{preset_id}_colab")
    client = LocalMCPClient(server)
    agent = LLMFullTypedMultiAgent(
        model=model,
        client=client,
        max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
        max_role_steps=MAX_ROLE_STEPS,
        max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
        max_parse_retries=MAX_PARSE_RETRIES,
        max_tool_retries=MAX_TOOL_RETRIES,
        temperature=TEMPERATURE,
        state_feedback_mode=STATE_FEEDBACK_MODE,
    )

    result = agent.run(query)
    result_dict = result.to_dict()

    gold = gold_runner.run(eval_task)
    metric_result = None
    gold_status = gold.status
    gold_error = gold.error
    gold_result = gold.result
    if gold.status == "success" and gold.result is not None:
        agent_metric_payload = dict(result.tool_results)
        agent_metric_payload.update({
            "parse_error_count": result.parse_error_count,
            "invalid_json_count": result.invalid_json_count,
            "invalid_role_protocol_count": result.invalid_role_protocol_count,
            "forbidden_tool_call_count": result.forbidden_tool_call_count,
            "repeated_tool_call_count": result.repeated_tool_call_count,
            "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
            "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
            "stalled_loop_detected": result.stalled_loop_detected,
        })
        metric_result = compare_agent_to_gold(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            agent_result=agent_metric_payload,
            gold_result=gold.result,
            agent_trace=result.trace,
            task=task_to_dict(eval_task),
            parsed_task=result.parsed_task,
            orchestration_steps=result.orchestration_steps,
        )

    metrics = metric_dict(metric_result)
    verdict_checks = build_verdict_checks(result, metric_result)
    overall_ok = overall_ok_from_checks(verdict_checks)

    record = {
        "selected_preset": preset_id,
        "task_config": task_config,
        "query": query,
        "expected_tool_sequence": expected_sequence,
        "actual_tool_sequence": result.actual_tool_sequence,
        "architecture": ARCHITECTURE_MODE,
        "model_name": MODEL_NAME,
        "typed_protocol_version": TYPED_PROTOCOL_VERSION,
        "prompt_revision": PROMPT_REVISION,
        "agent_result": result_dict,
        "gold_status": gold_status,
        "gold_error": gold_error,
        "gold_result": gold_result,
        "metrics": metrics,
        "verdict_checks": verdict_checks,
        "overall_ok": overall_ok,
        "success": bool(result.success),
        "final_answer_preview": final_answer_preview(result),
        "role_agent_order": result.role_agent_order,
        "orchestrator_decisions": result.orchestrator_decisions,
        "role_actions": result.role_actions,
        "role_assignments": result.role_assignments,
        "role_outputs": result.role_outputs,
        "tool_observations": result.tool_observations,
        "available_artifacts_snapshots": result.available_artifacts_snapshots,
        "invalid_assignment_count": result.invalid_assignment_count,
        "invalid_role_action_count": result.invalid_role_action_count,
        "invalid_role_response_count": result.invalid_role_response_count,
        "invalid_final_answer_count": result.invalid_final_answer_count,
        "invalid_tool_name_count": result.invalid_tool_name_count,
        "forbidden_tool_call_count": result.forbidden_tool_call_count,
        "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
        "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
        "premature_role_completion_count": result.premature_role_completion_count,
        "empty_findings_done_count": result.empty_findings_done_count,
        "repeated_equivalent_role_assignment_count": result.repeated_equivalent_role_assignment_count,
        "tool_error_count": result.tool_error_count,
        "tool_schema_validation_error_count": result.tool_schema_validation_error_count,
        "invalid_role_protocol_count": result.invalid_role_protocol_count,
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "repeated_tool_call_count": result.repeated_tool_call_count,
        "stalled_loop_detected": result.stalled_loop_detected,
        "repair_attempt_count": result.repair_attempt_count,
        "retry_count": result.retry_count,
        "recovery_attempt_count": result.recovery_attempt_count,
        "recovery_success_count": result.recovery_success_count,
        "recovery_failure_count": result.recovery_failure_count,
        "key_metric_summary": key_metric_summary(task_config["task_type"], metrics),
        "missing_agent_terminal_artifact": (metrics or {}).get("missing_agent_terminal_artifact"),
    }

    per_task_path = OUTPUT_DIR / PER_TASK_OUTPUT_TEMPLATE.format(preset_id=preset_id)
    with per_task_path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(record), f, ensure_ascii=False, indent=2)
    print("Saved:", per_task_path)
    print("success:", record["success"], "overall_ok:", overall_ok, "tools:", result.actual_tool_sequence)
    all_results.append(record)

## 10. Summary table and save batch

In [ ]:
def rate(values):
    values = [v for v in values if isinstance(v, bool)]
    return None if not values else sum(values) / len(values)

summary = {
    "n_tasks": len(all_results),
    "n_success": sum(1 for r in all_results if r["success"]),
    "n_failed": sum(1 for r in all_results if not r["success"]),
    "success_rate": rate([r["success"] for r in all_results]),
    "n_overall_ok": sum(1 for r in all_results if r["overall_ok"]),
    "overall_ok_rate": rate([r["overall_ok"] for r in all_results]),
    "agent_success_rate": rate([r["success"] for r in all_results]),
    "tool_sequence_match_rate": rate([(r.get("metrics") or {}).get("tool_sequence_match") for r in all_results]),
    "role_agent_order_match_rate": rate([(r.get("metrics") or {}).get("role_agent_order_match") for r in all_results]),
    "artifact_flow_valid_rate": rate([(r.get("metrics") or {}).get("artifact_flow_valid") for r in all_results]),
    "required_role_agents_called_rate": rate([(r.get("metrics") or {}).get("required_role_agents_called") for r in all_results]),
    "avg_tool_call_count": sum(len(r["actual_tool_sequence"]) for r in all_results) / max(1, len(all_results)),
    "avg_tool_error_count": sum(r["tool_error_count"] for r in all_results) / max(1, len(all_results)),
    "avg_parse_error_count": sum(r["parse_error_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_assignment_count": sum(r["invalid_assignment_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_role_response_count": sum(r["invalid_role_response_count"] for r in all_results) / max(1, len(all_results)),
    "avg_forbidden_tool_call_count": sum(r["forbidden_tool_call_count"] for r in all_results) / max(1, len(all_results)),
    "avg_multiple_protocol_blocks_in_single_output_count": sum(r["multiple_protocol_blocks_in_single_output_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_artifact_handle_count": sum(r["invalid_artifact_handle_count"] for r in all_results) / max(1, len(all_results)),
    "avg_premature_role_completion_count": sum(r["premature_role_completion_count"] for r in all_results) / max(1, len(all_results)),
    "avg_empty_findings_done_count": sum(r["empty_findings_done_count"] for r in all_results) / max(1, len(all_results)),
    "avg_repeated_equivalent_role_assignment_count": sum(r["repeated_equivalent_role_assignment_count"] for r in all_results) / max(1, len(all_results)),
    "avg_tool_schema_validation_error_count": sum(r["tool_schema_validation_error_count"] for r in all_results) / max(1, len(all_results)),
    "stalled_loop_rate": rate([r["stalled_loop_detected"] for r in all_results]),
    "legacy_report_tool_used_rate": rate([("tec_" + "build_report") in r["actual_tool_sequence"] for r in all_results]),
}

payload = {
    "experiment_id": EXPERIMENT_ID,
    "architecture": ARCHITECTURE_MODE,
    "model_name": MODEL_NAME,
    "dataset_ref": DATASET_REF,
    "dataset_path": str(DATASET_PATH),
    "model_config": {
        "model_name": MODEL_NAME,
        "load_in_4bit": LOAD_IN_4BIT,
        "load_in_8bit": LOAD_IN_8BIT,
        "torch_dtype": TORCH_DTYPE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": DO_SAMPLE,
    },
    "agent_config": {
        "max_orchestration_steps": MAX_ORCHESTRATION_STEPS,
        "max_role_steps": MAX_ROLE_STEPS,
        "max_tool_calls_per_role": MAX_TOOL_CALLS_PER_ROLE,
        "max_parse_retries": MAX_PARSE_RETRIES,
        "max_tool_retries": MAX_TOOL_RETRIES,
        "state_feedback_mode": STATE_FEEDBACK_MODE,
    },
    "typed_protocol_version": TYPED_PROTOCOL_VERSION,
    "prompt_revision": PROMPT_REVISION,
    "summary": summary,
    "results": all_results,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with BATCH_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(payload), f, ensure_ascii=False, indent=2)

summary_rows = []
for r in all_results:
    summary_rows.append({
        "preset_id": r["selected_preset"],
        "task_type": r["task_config"]["task_type"],
        "agent_success": r["success"],
        "overall_ok": r["overall_ok"],
        "tool_sequence_match": (r.get("metrics") or {}).get("tool_sequence_match"),
        "actual_tool_sequence": " -> ".join(r["actual_tool_sequence"]),
        "role_agent_order": " -> ".join(r["role_agent_order"]),
        "tool_call_count": len(r["actual_tool_sequence"]),
        "tool_error_count": r["tool_error_count"],
        "final_answer_present": bool((r["agent_result"] or {}).get("answer")),
        "role_agent_order_match": (r.get("metrics") or {}).get("role_agent_order_match"),
        "required_role_agents_called": (r.get("metrics") or {}).get("required_role_agents_called"),
        "artifact_flow_valid": (r.get("metrics") or {}).get("artifact_flow_valid"),
        "invalid_artifact_handle_count": r["invalid_artifact_handle_count"],
        "multiple_protocol_blocks_in_single_output_count": r["multiple_protocol_blocks_in_single_output_count"],
        "premature_role_completion_count": r["premature_role_completion_count"],
        "empty_findings_done_count": r["empty_findings_done_count"],
        "repeated_equivalent_role_assignment_count": r["repeated_equivalent_role_assignment_count"],
        "tool_schema_validation_error_count": r["tool_schema_validation_error_count"],
        "parse_error_count": r["parse_error_count"],
        "repeated_tool_call_count": r["repeated_tool_call_count"],
        "forbidden_tool_call_count": r["forbidden_tool_call_count"],
        "stalled_loop_detected": r["stalled_loop_detected"],
        "missing_agent_terminal_artifact": r.get("missing_agent_terminal_artifact"),
        "key_metric_summary": r["key_metric_summary"],
        "answer_preview": r["final_answer_preview"],
    })
print("Saved batch:", BATCH_OUTPUT_PATH)
pd.DataFrame(summary_rows)

## 11. Optional architecture comparison

This cell reads existing JSON files only. Missing sources are skipped gracefully.

In [ ]:
def load_optional(path):
    if not path.exists():
        print("Missing:", path)
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

comparison_sources = {
    "qwen_single": load_optional(OUTPUT_ROOT / "qwen_single_agent_batch_colab.json"),
    "rule_multi": load_optional(OUTPUT_ROOT / "multi_agent_rule_based_five_task_batch.json"),
    "qwen_multi_old": load_optional(OUTPUT_ROOT / "qwen_multi_agent_batch_colab.json"),
    "typed_v1_minimal_role_response": load_optional(OUTPUT_ROOT / "qwen_multi_agent_typed_batch_colab.json"),
    "typed_v2_role_boundaries_tool_schemas": load_optional(OUTPUT_ROOT / "experiment_3" / "qwen_multi_agent_typed_v2_batch_colab.json"),
    "typed_v3_grounded_inputs_deliverables_single_block": load_optional(BATCH_OUTPUT_PATH),
}

comparison_rows = []
for record in all_results:
    preset_id = record["selected_preset"]
    comparison_rows.append({
        "preset_id": preset_id,
        "task_type": record["task_config"]["task_type"],
        "typed_v3_success": record["success"],
        "typed_v3_stalled": record["stalled_loop_detected"],
        "typed_v3_role_order": " -> ".join(record["role_agent_order"]),
        "typed_v3_tool_sequence": " -> ".join(record["actual_tool_sequence"]),
        "typed_v3_invalid_artifact_handle_count": record["invalid_artifact_handle_count"],
        "typed_v3_multiple_protocol_blocks_in_single_output_count": record["multiple_protocol_blocks_in_single_output_count"],
        "typed_v3_premature_role_completion_count": record["premature_role_completion_count"],
        "typed_v3_empty_findings_done_count": record["empty_findings_done_count"],
        "typed_v3_repeated_equivalent_role_assignment_count": record["repeated_equivalent_role_assignment_count"],
        "typed_v3_tool_schema_validation_error_count": record["tool_schema_validation_error_count"],
    })
pd.DataFrame(comparison_rows)